# DSPy extract full

This NB focuses on extracting higher fidelity dialogues by incorporating:

* Plot description from Wiki
* Current page image
* Previous pages output
* Estimate for where in the plot we're at

The output should include:

* A description of the events taking place
* Dialogues with attribution
* New estimate for where in the plot we're at

In [3]:
from pathlib import Path

OUTPUT_DIR = Path("../output/dspy-extract-full")
INPUT_WIKI_ISSUES = sorted(list(Path("../output/wiki/fandom/crawl/storie/storie-di-pkna").glob("*.md")))
INPUT_ISSUES = sorted(list(Path("../input/pkna").glob("*")))

In [4]:
import os
from dotenv import load_dotenv
import dspy


load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-pro',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_tokens=65535,
)
dspy.configure(lm=lm, track_usage=True)

lm("Who are you?")  # Warm up the model

['I am a large language model, trained by Google.']

In [ ]:
from dataclasses import dataclass

import dspy
from pydantic import BaseModel, Field


class DialogueLine(BaseModel):
    """A single line of dialogue spoken by a character."""

    character: str = Field(description="The name of the character speaking the line.")
    line: str = Field(description="The dialogue line spoken by the character.")


class Panel(BaseModel):
    """A single panel from a comic book page."""

    description: str = Field(
        description="A detailed description of the events happening in the panel."
    )
    caption_text: str | None = Field(
        default=None,
        description="The text of any caption present in the panel, if applicable.",
    )
    dialogues: list[DialogueLine] = Field(
        description="A list of dialogue lines spoken by characters in the panel. Dialogues are in order of appearance."
    )


class PlotExtractor(dspy.Signature):
    """Take a comic book issue summary and extract structured information about its plot.

    Follow these instructions:
    - Identify the main characters involved in the plot.
    - Summarize the key events that drive the story forward.
    - Use the language of the comic book (Italian) for all summaries and descriptions.
    """

    issue_summary: str = dspy.InputField(
        description="A brief summary of the comic book issue."
    )

    main_characters: list[str] = dspy.OutputField(
        description="A list of main characters involved in the plot."
    )
    key_events: list[str] = dspy.OutputField(
        description="A list of key events that drive the story forward."
    )


class PageExtractor(dspy.Signature):
    """Take a comic book page image and extract structured information about its content.

    FOLLOW THESE IMPORTANT INSTRUCTIONS

    Story continuity:
    - Use the overall plot summary and key events to inform the page content.
    - Use the previous page summary and panels to maintain continuity.
    - Use the same character names as previously introduced, if the character is the same.
    - New character can be introduced if they haven't appeared before.

    Last event tracking:
    - Use the last event to keep track of story progression.
    - Update the last event if a new key event occurs on this page.
    - In the last event, refer to the key events by their exact wording as provided. DO NOT invent new key events.

    Character attribution:
    - Make extra effort to correctly attribute dialogue lines to the right characters.
    - The speaker of a line might not always be visible in the panel. Use context from previous and following panels to infer the speaker.

    Output text:
    - Use the language of the comic book (Italian) for all summaries and descriptions.
    - Normalize the text by using normal caps instead of all caps, remove line-break hyphens, and accented letters instead of apostrophes when appropriate.
    """

    page: dspy.Image = dspy.InputField(
        description="The comic book page image to be analyzed."
    )
    previous_page_summary: str | None = dspy.InputField(
        default=None,
        description="A brief summary of the previous comic book page, if available.",
    )
    previous_page_panels: list[Panel] | None = dspy.InputField(
        default=None,
        description="The panels from the previous comic book page, if available.",
    )
    characters_already_introduced: list[str] = dspy.InputField(
        default=[],
        description="A list of character names that have already been introduced in previous pages.",
    )
    plot_summary: str = dspy.InputField(
        description="A brief summary of the overall plot of the comic book issue."
    )
    key_events: list[str] = dspy.InputField(
        description="A list of the key events in the overall story of the comic book issue."
    )
    last_event: str | None = dspy.InputField(
        default=None,
        description="The last key event that occurred in the comic book issue, if available.",
    )

    summary: str = dspy.OutputField(
        description="A brief summary of the comic book page."
    )
    panels: list[Panel] = dspy.OutputField(
        description="A list of panels extracted from the comic book page."
    )
    new_last_event: str | None = dspy.OutputField(
        default=None,
        description="The last key event that occurred in the comic book issue after this page, if available.",
    )


@dataclass
class IssueSummary:
    summary: str
    key_events: list[str]
    main_characters: list[str]


class PlotSummarizer(dspy.Module):
    def __init__(self):
        self.plot_extractor = dspy.ChainOfThought(PlotExtractor)

    def forward(self, issue_summary: str) -> dspy.Prediction:
        pred = self.plot_extractor(issue_summary=issue_summary)
        pred.issue_summary = issue_summary  # Store original summary
        return pred

    def to_dataclass(self, pred: dspy.Prediction) -> IssueSummary:
        return IssueSummary(
            summary=pred.issue_summary,
            key_events=pred.key_events,
            main_characters=pred.main_characters,
        )


class Extractor(dspy.Module):
    def __init__(self, summary: IssueSummary):
        self.issue_summary = summary

        self.prev_characters = set[str]()
        self.prev_page_summary: str | None = None
        self.prev_page_panels: list[Panel] | None = None
        self.last_event: str | None = None

        self.plot_extractor = dspy.ChainOfThought(PlotExtractor)
        self.page_extractor = dspy.ChainOfThought(PageExtractor)

    def forward(self, page: dspy.Image) -> dspy.Prediction:
        page_pred = self.page_extractor(
            page=page,
            previous_page_summary=self.prev_page_summary,
            previous_page_panels=self.prev_page_panels,
            characters_already_introduced=list(self.prev_characters),
            plot_summary=self.issue_summary.summary,
            key_events=self.issue_summary.key_events,
            last_event=self.last_event,
        )
        # Update state for next page
        self.prev_page_summary = page_pred.summary
        self.prev_page_panels = page_pred.panels
        self.prev_characters.update(
            [dialogue.character for panel in page_pred.panels for dialogue in panel.dialogues]
        )
        self.last_event = page_pred.new_last_event

        return page_pred


## Vibe-check

Use Mekkano to check quality.

In [6]:
pages = [
    dspy.Image.from_file(x.as_posix())
    for x in sorted(Path("../input/pkna/pkna-20").glob("*.jpeg"))
]
with open("../output/wiki/fandom/crawl/storie/storie-di-pkna/mekkano.md") as f:
    plot = f.read()

In [7]:
len(pages), len(plot)

(62, 5671)

In [8]:
summarizer = PlotSummarizer()
pred = summarizer(issue_summary=plot)
summary = summarizer.to_dataclass(pred)
summary

IssueSummary(summary="# Mekkano\n\nMekkano è un episodio della serie PKNA,\xa0 scritto da Bruno Enna, disegnato da Andrea Freccero e pubblicato per la prima volta nell'agosto 1998 sul numero 20 della rivista omonima; è incentrato sulla figura dell'evroniano Gorthan e sul suo odio-amore per la cultura terrestre.\n\n## Trama\n\nGorthan è stato condannato a morte dai suoi compagni evroniani, dopo che, in seguito ai suoi contatti con i terrestri, in lui si è sviluppata la capacità di provare emozioni. Costretto alla fuga, precipita con la sua astronave sulla Terra; nella caduta, è costretto a gettare il suo serbatoio di energia vitale.\n\nLo schianto avviene nel Wolf Canyon, dove è in corso la dimostrazione di un nuovo robot per l’esplorazione spaziale,\xa0 fabbricato dall'ingegner Takeda delle Animated Industries. Gorthan, indebolito dalla mancanza di energie emozionali, è catturato da PK, prontamente accorso sulla Pi-kar, e portato nei sotterranei della Ducklair Tower. Uno diagnostica ch

In [9]:
extractor = Extractor(summary=summary)

In [10]:
page1 = extractor(page=pages[0])
page1

Prediction(
    reasoning='La pagina mostra una singola grande vignetta che raffigura l\'inizio della storia, come descritto nel riassunto della trama e nel primo evento chiave. L\'alieno visibile è Gorthan, un evroniano, che sta precipitando sulla Terra con la sua astronave. I dialoghi provengono dal sistema di autoguida della sua nave, che chiamo "Gravithar", come specificato nel testo. L\'ultima casella di testo è un pensiero di Gorthan, che tratto come una didascalia. Questa pagina corrisponde pienamente al primo evento chiave: "Gorthan, condannato a morte dagli Evroniani per aver sviluppato emozioni, fugge e precipita sulla Terra, perdendo il suo serbatoio di energia vitale.". Pertanto, imposterò questo evento come il "new_last_event". Introdurrò i personaggi Gorthan e Gravithar.',
    summary="Mentre la sua astronave in fiamme precipita verso la Terra, il sistema di autoguida Gravithar informa il Capo-Branca Gorthan che l'impatto con il suolo è imminente. Gorthan, sconvolto tra i

In [11]:
page2 = extractor(page=pages[1])
page2

Prediction(
    reasoning='La pagina analizzata prosegue la scena iniziata nella pagina precedente: l\'astronave del Capo-Branca Gorthan sta precipitando sulla Terra.\nIdentifico i personaggi e le loro azioni. Il personaggio principale è Gorthan, l\'evroniano nel suo scafandro da battaglia. L\'altra entità che parla è Gravithar, l\'intelligenza artificiale della nave, già introdotta.\nIl dialogo è diviso tra Gravithar, che fornisce aggiornamenti tecnici sulla situazione disperata (danni non quantificabili, sistema di espulsione fuori uso, via di fuga troppo stretta), e Gorthan, che recita dei versi poetici. Dal riassunto della trama so che queste sono citazioni da "Il piccolo principe", un dettaglio cruciale per lo sviluppo del suo personaggio.\nL\'azione principale è la conseguenza delle informazioni di Gravithar: per passare attraverso lo squarcio nella fusoliera, Gorthan è costretto a espellere il suo "serbatoio di energia vitale". Questo evento è centrale per la trama, come indicat

In [12]:
import json

vibe_out_path = OUTPUT_DIR / "vibe"
vibe_out_path.mkdir(parents=True, exist_ok=True)


def save_vibe_prediction(page_pred: dspy.Prediction, index: int) -> None:
    data = {
        "reasoning": page_pred.reasoning,
        "summary": page_pred.summary,
        "panels": [
            {
                "description": panel.description,
                "caption_text": panel.caption_text,
                "dialogues": [
                    {"character": dlg.character, "line": dlg.line}
                    for dlg in panel.dialogues
                ],
            }
            for panel in page_pred.panels
        ],
        "last_event": page_pred.new_last_event,
    }
    with open(vibe_out_path / f"pkna-20-{index}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


save_vibe_prediction(page1, index=0)
save_vibe_prediction(page2, index=1)